<a href="https://colab.research.google.com/github/Realbenjizo/python-functions-for-data-engineers/blob/main/Chapter_8_Lazy_Evaluation_with_Generators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import logging

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

In [ ]:
numbers = [i * 2 for i in range(1_000_000)]

In [ ]:
def generate_numbers(n: int=1_000_000):
    for i in range(n):
        yield i * 2

In [ ]:
import pandas as pd

# Replace with link to your dowonloaded file
data = "/content/drive/MyDrive/data_files/transactions_2.csv"

# Creating a DataFrame
transactions_df = pd.read_csv(data)

# Parse the transaction_date column
transactions_df['transaction_date']=pd.to_datetime(transactions_df['transaction_date'])
print(transactions_df.head())

   user_id      amount    transaction_date     status
0      4.0  367.154980 2025-01-01 00:00:00  completed
1      2.0   42.518694 2025-01-01 00:01:00     failed
2      3.0  204.574847 2025-01-01 00:02:00     failed
3      3.0  315.463172 2025-01-01 00:03:00  completed
4      2.0  346.918579 2025-01-01 00:04:00     failed


In [ ]:
from typing import Dict

user_lookup: Dict[int, str] = {
    1: "Alice Johnson",
    2: "Bob Smith",
    3: "Carol Williams",
    4: "David Brown"
}

In [ ]:
from typing import Iterator, Dict, Any
import pandas as pd

Transaction = Dict[str, Any]
def generate_transactions(df: pd.DataFrame):
    """Yield one row at a time as a dictionary (lazy processing)."""
    for _, row in df.iterrows():
        yield {
            'user_id': row['user_id'],
            'amount': row['amount'],
            'transaction_date': row['transaction_date'],
            'status': row['status']
        }

# Creating a generator
results = generate_transactions(transactions_df)

In [ ]:
next(results)

{'user_id': 4.0,
 'amount': 367.1549798190639,
 'transaction_date': Timestamp('2025-01-01 00:00:00'),
 'status': 'completed'}

In [ ]:
next(results)

{'user_id': 2.0,
 'amount': 42.51869414213338,
 'transaction_date': Timestamp('2025-01-01 00:01:00'),
 'status': 'failed'}

In [ ]:
from typing import Iterator, Dict, Any
import pandas as pd

Transaction = Dict[str, Any]

def generate_transactions(df: pd.DataFrame) -> Iterator[Transaction]:
    """Yield one row at a time as a dictionary (lazy processing)."""
    for _, row in df.iterrows():
        yield {
            'user_id': row['user_id'],
            'amount': row['amount'],
            'transaction_date': row['transaction_date'],
            'status': row['status']
        }

# Creating a generator
results = generate_transactions(transactions_df)

## **Why Generators are Useful**

In [ ]:
import pandas as pd
from typing import Iterator, Dict, Any

Transaction = Dict[str, Any]
def generate_transactions_lazy(df: pd.DataFrame) -> Iterator[Transaction]:
    """Yield one row at a time as a dictionary (lazy processing)."""
    for _, row in df.iterrows():
        yield {
            'user_id': row['user_id'],
            'amount': row['amount'],
            'transaction_date': row['transaction_date'],
            'status': row['status']
        }

# Step 2
def filter_valid_transactions_lazy(
    transactions: Iterator[Transaction]
) -> Iterator[Transaction]:
    """Lazily filter out invalid transactions."""
    for tx in transactions:
        if pd.isna(tx['user_id']) or tx['amount'] <= 0:
            continue
        yield tx

# Step 3
def enrich_with_user_name_lazy(
    transactions: Iterator[Transaction],
    user_lookup: Dict[int, str]
) -> Iterator[Transaction]:
    """Lazily add user names."""
    for tx in transactions:
        tx['user_name'] = user_lookup.get(tx['user_id'], 'Unknown')
        yield tx

# Run the lazy pipeline
pipeline = generate_transactions_lazy(transactions_df)
pipeline = filter_valid_transactions_lazy(pipeline)
pipeline = enrich_with_user_name_lazy(pipeline, user_lookup)

if __name__ == "__main__":
    for transaction in pipeline:
         print(transaction)

Streaming output truncated to the last 5000 lines.
{'user_id': 2.0, 'amount': 80.085463615722, 'transaction_date': Timestamp('2025-12-09 10:47:00'), 'status': 'failed', 'user_name': 'Bob Smith'}
{'user_id': 3.0, 'amount': 310.9290299714855, 'transaction_date': Timestamp('2025-12-09 10:48:00'), 'status': 'completed', 'user_name': 'Carol Williams'}
{'user_id': 3.0, 'amount': 296.6482077454683, 'transaction_date': Timestamp('2025-12-09 10:49:00'), 'status': 'completed', 'user_name': 'Carol Williams'}
{'user_id': 2.0, 'amount': 46.85272200654916, 'transaction_date': Timestamp('2025-12-09 10:51:00'), 'status': 'failed', 'user_name': 'Bob Smith'}
{'user_id': 3.0, 'amount': 15.109407130962552, 'transaction_date': Timestamp('2025-12-09 10:52:00'), 'status': 'pending', 'user_name': 'Carol Williams'}
{'user_id': 1.0, 'amount': 302.0945712302487, 'transaction_date': Timestamp('2025-12-09 10:53:00'), 'status': 'completed', 'user_name': 'Alice Johnson'}
{'user_id': 4.0, 'amount': 39.707426761245, '

# **Eager Pipeline**

In [ ]:
import pandas as pd
from typing import Dict, List
import logging

logger = logging.getLogger(__name__)

def generate_transactions_eager(df: pd.DataFrame) -> List[Dict]:
    return [
        {
            'user_id': row['user_id'],
            'amount': row['amount'],
            'transaction_date': row['transaction_date'],
            'status': row['status']
        }
        for _, row in df.iterrows()
    ]

def filter_valid_transactions_eager(transactions: List[Dict]) -> List[Dict]:
    return [
        tx for tx in transactions
        if not pd.isna(tx['user_id']) and tx['amount'] > 0
    ]

def enrich_with_user_name_eager(
    transactions: List[Dict],
    user_lookup: Dict[int, str]
) -> list[dict]:
    return [
        {**tx, 'user_name': user_lookup.get(tx['user_id'], 'Unknown')}
        for tx in transactions
    ]

# Executing the pipeline
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)

    # Step 1: Generates ALL transactions at once
    transactions = generate_transactions_eager(transactions_df)

    # Step 2: Filters ALL transactions at once and returns all filtered transactions
    valid_transactions = filter_valid_transactions_eager(transactions)

    # Step 3: Enriches ALL transactions at once and returns all enriched transactions
    enriched = enrich_with_user_name_eager(valid_transactions, user_lookup)

    logger.info(f"Processed {len(enriched)} transactions")
    print("\nSample results:")
    for tx in enriched[:3]:
        print(f"  {tx}")

INFO:__main__:Processed 363669 transactions



Sample results:
  {'user_id': 4.0, 'amount': 367.1549798190639, 'transaction_date': Timestamp('2025-01-01 00:00:00'), 'status': 'completed', 'user_name': 'David Brown'}
  {'user_id': 2.0, 'amount': 42.51869414213338, 'transaction_date': Timestamp('2025-01-01 00:01:00'), 'status': 'failed', 'user_name': 'Bob Smith'}
  {'user_id': 3.0, 'amount': 204.5748469893245, 'transaction_date': Timestamp('2025-01-01 00:02:00'), 'status': 'failed', 'user_name': 'Carol Williams'}


# **Memory Comparizon**

In [ ]:
import tracemalloc

# Compare eager pipeline
tracemalloc.start()
transactions = generate_transactions_eager(transactions_df)
valid_transactions = filter_valid_transactions_eager(transactions)
enriched = enrich_with_user_name_eager(valid_transactions, user_lookup)
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Eager pipeline - Peak memory: {peak / 1024 / 1024:.2f} MB")

# Compare lazy pipeline
tracemalloc.start()
pipeline = generate_transactions_lazy(transactions_df)
pipeline = filter_valid_transactions_lazy(pipeline)
pipeline = enrich_with_user_name_lazy(pipeline, user_lookup)
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Lazy pipeline - Peak memory: {peak / 1024 / 1024:.2f} MB")

Eager pipeline - Peak memory: 241.23 MB
Lazy pipeline - Peak memory: 0.02 MB


#  **Challenge**

In [ ]:
import pandas as pd

def load_transactions_eager(df: pd.DataFrame) -> list:
    """Load all transactions into a list (eager)."""
    transactions = []
    for _, row in df.iterrows():
        transactions.append(row.to_dict())
    return transactions

# DataFrame
df = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5],
    'amount': [100, 250, 50, 500, 200],
    'status': ['completed', 'pending', 'completed', 'failed', 'completed']
})

# Loads everything at once (Eager)
all_transactions = load_transactions_eager(df)
print(f"Loaded {len(all_transactions)} transactions")

Loaded 5 transactions


# **Solution**

In [ ]:
import pandas as pd

def generate_transactions_lazy(df: pd.DataFrame):
    """Yield transactions one at a time (lazy)."""
    for _, row in df.iterrows():
        yield row.to_dict()


df = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5],
    'amount': [100, 250, 50, 500, 200],
    'status': ['completed', 'pending', 'completed', 'failed', 'completed']
})


In [ ]:
def generate_transactions_lazy(df: pd.DataFrame):
    """Yield transactions one at a time (lazy)."""
    for _, row in df.iterrows():
        yield row.to_dict()

# Calling the function
if __name__ == "__main__":
    result = generate_transactions_lazy(df)
    for tx in result:
        print(tx)

{'user_id': 1, 'amount': 100, 'status': 'completed'}
{'user_id': 2, 'amount': 250, 'status': 'pending'}
{'user_id': 3, 'amount': 50, 'status': 'completed'}
{'user_id': 4, 'amount': 500, 'status': 'failed'}
{'user_id': 5, 'amount': 200, 'status': 'completed'}
